In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [185]:
products = spark.sparkContext.textFile('/content/part-00000', 10).map(lambda line: line.split(',')).filter(lambda x: x[4] != '').cache()
products.count()

1344

In [186]:
products.take(2)

[['1',
  '2',
  'Quest Q64 10 FT. x 10 FT. Slant Leg Instant U',
  '',
  '59.98',
  'http://images.acmesports.sports/Quest+Q64+10+FT.+x+10+FT.+Slant+Leg+Instant+Up+Canopy'],
 ['2',
  '2',
  "Under Armour Men's Highlight MC Football Clea",
  '',
  '129.99',
  'http://images.acmesports.sports/Under+Armour+Men%27s+Highlight+MC+Football+Cleat']]

## Global Ranking

Global ranking assigns a rank to each element across the entire dataset.


#### Extract top 10 products based on price
- First, create key-value pairs where the key is the ranking criterion (Price)
- Sort using sortByKey() to order elements

In [187]:
def keyByPrice(index, iter):
  for line in iter:
    yield (float(line[4]), (int(line[0]), int(line[1])))

In [188]:
print('Price, ID, Category')
for i in products.mapPartitionsWithIndex(keyByPrice).sortByKey(ascending=False).take(10): print(f'{i[0]}, {i[1][0]}, {i[1][1]}')

Price, ID, Category
1999.99, 208, 10
1799.99, 66, 4
1799.99, 199, 10
1799.99, 496, 22
1099.99, 1048, 47
999.99, 60, 4
999.99, 197, 10
999.99, 488, 22
999.99, 694, 32
999.99, 695, 32


# Per-Group Ranking
#### Get top 2 priced products per category
- First, create key-value pairs where the key is the category
- Use groupByKey() to group products with same category
- Then sort the records based on price and pick only 2 records per category

In [189]:
def keyByCategory(iter):
  for line in iter:
    yield (int(line[1]), (int(line[0]), float(line[4])))

In [191]:
print('Category, ID, Price')
for i in products.mapPartitions(keyByCategory).groupByKey().flatMapValues(lambda x: sorted(x, key=lambda k: k[1], reverse=True)[:2]).take(10): print(f'{i[0]}, {i[1][0]}, {i[1][1]}')

Category, ID, Price
10, 208, 1999.99
10, 199, 1799.99
20, 450, 249.99
20, 452, 169.99
30, 663, 139.99
30, 664, 139.99
40, 885, 24.99
40, 886, 24.99
50, 1116, 130.0
50, 1117, 130.0
